In [1]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
fake = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/Fake.csv")
real = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/True.csv")

fake["label"] = 0   # Fake = 0
real["label"] = 1   # Real = 1

df = pd.concat([fake, real])
df = df.sample(frac=1).reset_index(drop=True)

df.head()


,title,text,subject,date,label
0,WATCH: Muslim Journalist Absolutely DESTROYS ...,Donald Trump s spokeswoman Katrina Pierson wen...,News,"March 12, 2016",0
1,Argentine navy says unusual noise heard on day...,BUENOS AIRES (Reuters) - An unusual noise was ...,worldnews,"November 22, 2017",1
2,Saudi king says kingdom has made progress in t...,"MECCA, Saudi Arabia (Reuters) - Saudi King Sal...",worldnews,"September 2, 2017",1
3,Trump to give speech on illegal immigration on...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"August 28, 2016",1
4,Trump Swings Back at Author of Fake Dossier: ‘...,"21st Century Wire says In a Tweet, Donald Trum...",US_News,"January 19, 2017",0


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text


In [4]:
df["text"] = df["text"].apply(clean_text)


In [5]:
X = df["text"]
y = df["label"]

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(X)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


In [7]:
model = LogisticRegression()
model.fit(X_train, y_train)


LogisticRegression()

In [8]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9861915367483296

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      5968
           1       0.98      0.99      0.99      5257

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [9]:
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)


[[5879   89]
 [  66 5191]]


In [10]:
def predict_news(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)
    
    return "REAL NEWS ✅" if prediction[0] == 1 else "FAKE NEWS ❌"


In [12]:
print(predict_news("The Reserve Bank of India increased the repo rate by 25 basis points on Friday."))


REAL NEWS ✅


In [13]:
print(predict_news("Aliens landed in Delhi last night and offered free unlimited electricity to all citizens."))


FAKE NEWS ❌
